# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR² Mass Spectrometry Extracellular Vesicles Dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library and Python data tools.

### Dataset Source
The dataset is described by a [Croissant schema (JSON-LD)](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure mlcroissant is installed (latest version recommended)
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title : {metadata.name}\n")
print(f"Description  : {metadata.description}\n")
print(f"Version      : {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, a 'record set' is a tabular or structured group of entries (like a primary data table). Fields correspond to columns. All entities are referenced by their canonical `@id`.

Let's list all record sets, their IDs, and their available fields (columns):

In [ ]:
# List all record sets, their @id and fields
print("Record sets in dataset:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    print(f"  Fields (@id):")
    for f in rs.fields:
        print(f"    - {f.name} (@id: {f.id}, type: {getattr(f, 'data_type', 'unknown')})")
    print()

Below, we show a preview of the first record in a selected record set. (Change `<record_set_id>` to one of the IDs above.)

In [ ]:
# Pick a record set (by @id)
# In this dataset, let's use the first available record set for demonstration.
record_set_ids = [rs.id for rs in record_sets]
selected_record_set_id = record_set_ids[0]
print(f"Sample record from RecordSet '{selected_record_set_id}':\n")
# Print one record as a dict
for i, record in enumerate(dataset.records(record_set=selected_record_set_id)):
    print(record)
    if i >= 0:
        break

## 3. Data Extraction
Load data from each record set into separate DataFrames for analysis, referencing all table and field identifiers by their `@id`.

In [ ]:
# Extract each record set into its own dataframe
dfs = {}  # Map of record set @id to DataFrame
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dfs[rs.id] = df
    print(f"RecordSet '{rs.name}' (@id: {rs.id}) loaded, shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}\n")
# For demonstration, show a sample of the first record set
display_id = record_set_ids[0]
print(f"First few rows from '{display_id}':")
dfs[display_id].head()

## 4. Exploratory Data Analysis (EDA)
Now, let's apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data for summary statistics.

We'll:
1. Select a numeric field (referenced by its `@id`) from the loaded DataFrame.
2. Filter records where the field is above a chosen threshold.
3. Normalize this field.
4. Group by a categorical field if available.

In [ ]:
df = dfs[display_id]
# Choose a numeric field and a group field.
# For demonstration: Find the first float/integer field as numeric, and next as groupable (string/categorical)
numeric_field_id = None
group_field_id = None
for f in [f for f in record_sets[0].fields]:
    dt = (getattr(f, 'data_type', '') or '').lower()
    if numeric_field_id is None and dt in ['float', 'integer', 'number'] and f.id in df.columns:
        numeric_field_id = f.id
    if group_field_id is None and dt in ['string', 'text'] and f.id in df.columns:
        group_field_id = f.id
if numeric_field_id is None:
    raise Exception("No numeric field found!")
if group_field_id is None:
    # fallback: use the first non-numeric field
    for f in record_sets[0].fields:
        dt = (getattr(f, 'data_type', '') or '').lower()
        if dt not in ['float', 'integer', 'number'] and f.id in df.columns:
            group_field_id = f.id
            break
print(f"Numeric field: {numeric_field_id}\nGroup field: {group_field_id}")
if numeric_field_id not in df.columns:
    raise Exception(f"Field '{numeric_field_id}' not in dataframe columns!")

# Convert numeric field to float
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Set threshold (use median if possible)
threshold = df[numeric_field_id].median() if not df[numeric_field_id].isnull().all() else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered rows where {numeric_field_id} > {threshold:.3f} (count={len(filtered_df)}):\n")
print(filtered_df.head())

# Normalize the numeric column
col_norm = numeric_field_id + "_normalized"
filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized field added: '{col_norm}'\n")
print(filtered_df[[numeric_field_id, col_norm]].head())

# Group by the group field (if available)
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(by=numeric_field_id, ascending=False)
    print(f"\nMean {numeric_field_id} grouped by '{group_field_id}':")
    print(grouped_df.head())

## 5. Visualization
We can visualize data distributions and relationships between numeric and categorical fields. Let's plot a histogram of the selected numeric field, and a bar chart by the group field if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True, color='skyblue')
plt.title(f"Distribution of '{numeric_field_id}'")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Barplot by group (if grouped_df is available)
if 'grouped_df' in locals() and isinstance(grouped_df, pd.DataFrame) and len(grouped_df) > 0:
    plt.figure(figsize=(10,4))
    sns.barplot(
        data=grouped_df.head(20),
        x=group_field_id, y=numeric_field_id, palette='viridis')
    plt.title(f"Mean of '{numeric_field_id}' by '{group_field_id}' (Top 20)")
    plt.xticks(rotation=60)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading, exploring, and analyzing the Mass Spectrometry Analysis of Extracellular Vesicles dataset (referenced entirely using Croissant `@id` URIs). Key steps included:
- Accessing rich metadata and structure via Croissant
- Exploring tables and fields by their `@id`
- Extracting tabular data to Pandas DataFrames
- Performing basic filtering, normalization, grouping, and visualization

This approach allows for robust, FAIR-compliant data science that transparently tracks field and table provenance.